In [0]:
# Config from ../04_utils/shared_config (inlined because %run path resolution fails on serverless)
CATALOG = "workspace"
SCHEMA = "crypto_live"
VOLUME_NAME = "raw_landing"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

from pyspark.sql.functions import col, from_unixtime, current_timestamp

# Auto Loader — the production-grade way to handle "new files keep arriving incrementally"
# Instead of re-reading everything every run, it tracks which files it's already processed
bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{VOLUME_PATH}/_schema_bronze")   # Auto Loader tracks schema here
    .load(VOLUME_PATH)
)

# Write as a Bronze Delta table, appending new records as they arrive
(
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", f"{VOLUME_PATH}/_checkpoint_bronze")     # tracks WHICH files were already processed
    .trigger(availableNow=True)    # process everything currently available, then stop (batch-style, fits a scheduled job)
    .toTable(f"{CATALOG}.{SCHEMA}.bronze_crypto_prices")
)

print("Bronze ingestion complete (Auto Loader).")
display(spark.table(f"{CATALOG}.{SCHEMA}.bronze_crypto_prices").limit(5))

Bronze ingestion complete (Auto Loader).


_snapshot_timestamp,binancecoin,bitcoin,cardano,dogecoin,ethereum,litecoin,polkadot,ripple,solana,tether,_rescued_data
2026-09-20T15:55:59.602605+00:00,"{""usd"":756.39,""usd_market_cap"":1.007170844415479E11,""usd_24h_vol"":7.940474626420232E8,""usd_24h_change"":-1.4875527168571128,""last_updated_at"":1789919670}","{""usd"":80867,""usd_market_cap"":1.6243305455395098E12,""usd_24h_vol"":2.2987854727548573E10,""usd_24h_change"":-0.9546039276867199,""last_updated_at"":1789919660}","{""usd"":0.224001,""usd_market_cap"":8.403469286011853E9,""usd_24h_vol"":4.1712855896768E8,""usd_24h_change"":-1.3301570872242114,""last_updated_at"":1789919650}","{""usd"":0.085594,""usd_market_cap"":1.3351168886484896E10,""usd_24h_vol"":9.335344313024957E8,""usd_24h_change"":-3.762366851257769,""last_updated_at"":1789919670}","{""usd"":2606.44,""usd_market_cap"":3.1815178829381085E11,""usd_24h_vol"":1.0139341618145624E10,""usd_24h_change"":-1.3768780461985037,""last_updated_at"":1789919660}","{""usd"":57.38,""usd_market_cap"":4.453892393366436E9,""usd_24h_vol"":2.717193944390818E8,""usd_24h_change"":-1.1010088557509428,""last_updated_at"":1789919680}","{""usd"":1.11,""usd_market_cap"":1.884625404139482E9,""usd_24h_vol"":1.7863282914742526E8,""usd_24h_change"":-1.4709416282824916,""last_updated_at"":1789919650}","{""usd"":1.39,""usd_market_cap"":8.724188117026244E10,""usd_24h_vol"":2.4372103155386252E9,""usd_24h_change"":-3.1116037809210493,""last_updated_at"":1789919680}","{""usd"":108.51,""usd_market_cap"":6.3729090472030235E10,""usd_24h_vol"":2.7748081671470056E9,""usd_24h_change"":-2.797511923601268,""last_updated_at"":1789919650}","{""usd"":0.999621,""usd_market_cap"":1.8331506967557727E11,""usd_24h_vol"":4.6594457253287285E10,""usd_24h_change"":-5.448846133229821E-4,""last_updated_at"":1789919670}",null


In [0]:
from pyspark.sql.functions import explode, map_keys, col, from_json

# Config from ../04_utils/shared_config (inlined because %run path resolution fails on serverless)
COIN_IDS = [
    "bitcoin", "ethereum", "tether", "binancecoin", "solana",
    "ripple", "cardano", "dogecoin", "polkadot", "litecoin"
]

bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_crypto_prices")

# The raw JSON has each coin as a nested object (e.g., {"bitcoin": {"usd": 65000, ...}})
# This flattens it into one row per coin per snapshot
coin_columns = [c for c in bronze.columns if c not in ("_snapshot_timestamp", "_rescued_data", "_file_path")]

rows = []
for coin in COIN_IDS:
    if coin in bronze.columns:
        coin_struct = from_json(col(coin), "struct<usd: double, usd_market_cap: double, usd_24h_vol: double, usd_24h_change: double, last_updated_at: long>")
        silver_partial = bronze.select(
            coin_struct.getField("usd").alias("price_usd"),
            coin_struct.getField("usd_market_cap").alias("market_cap_usd"),
            coin_struct.getField("usd_24h_vol").alias("volume_24h_usd"),
            coin_struct.getField("usd_24h_change").alias("change_24h_pct"),
            coin_struct.getField("last_updated_at").alias("last_updated_unix"),
            col("_snapshot_timestamp")
        ).withColumn("coin_id", col("_snapshot_timestamp").substr(1, 0))  # placeholder, fixed below

        # simpler, correct approach: add coin_id as a literal
        from pyspark.sql.functions import lit
        silver_partial = silver_partial.withColumn("coin_id", lit(coin))
        rows.append(silver_partial)

silver_df = rows[0]
for r in rows[1:]:
    silver_df = silver_df.unionByName(r)

silver_df = (
    silver_df
    .dropna(subset=["price_usd", "coin_id"])
    .withColumn("last_updated_ts", from_unixtime(col("last_updated_unix")))
    .select("coin_id", "price_usd", "market_cap_usd", "volume_24h_usd", "change_24h_pct", "last_updated_ts", "_snapshot_timestamp")
)

silver_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_crypto_prices")

print(f"Silver rows added: {silver_df.count()}")
display(silver_df)

Silver rows added: 20


coin_id,price_usd,market_cap_usd,volume_24h_usd,change_24h_pct,last_updated_ts,_snapshot_timestamp
bitcoin,80867.0,1.6243305455395098E12,2.2987854727548573E10,-0.9546039276867199,2026-09-20 15:54:20,2026-09-20T15:55:59.602605+00:00
bitcoin,81134.0,1.629742590481104E12,2.3188854198653336E10,-0.7422942528515162,2026-09-20 16:16:10,2026-09-20T16:18:06.112233+00:00
ethereum,2606.44,3.1815178829381085E11,1.0139341618145624E10,-1.3768780461985037,2026-09-20 15:54:20,2026-09-20T15:55:59.602605+00:00
ethereum,2622.88,3.2016170274783344E11,1.0379534242344746E10,-0.7591569457511214,2026-09-20 16:16:10,2026-09-20T16:18:06.112233+00:00
tether,0.999621,1.8331506967557727E11,4.6594457253287285E10,-5.448846133229821E-4,2026-09-20 15:54:30,2026-09-20T15:55:59.602605+00:00
tether,0.999643,1.8332243340345624E11,4.7157997119604744E10,5.813516019974668E-4,2026-09-20 16:16:20,2026-09-20T16:18:06.112233+00:00
binancecoin,756.39,1.007170844415479E11,7.940474626420232E8,-1.4875527168571128,2026-09-20 15:54:30,2026-09-20T15:55:59.602605+00:00
binancecoin,759.19,1.0109457615424792E11,9.038363893854249E8,-0.9813460267508477,2026-09-20 16:16:10,2026-09-20T16:18:06.112233+00:00
solana,108.51,6.3729090472030235E10,2.7748081671470056E9,-2.797511923601268,2026-09-20 15:54:10,2026-09-20T15:55:59.602605+00:00
solana,108.87,6.394948211442102E10,2.917866665509385E9,-2.451591934282332,2026-09-20 16:16:10,2026-09-20T16:18:06.112233+00:00
